# Middleware

## Summarization Middleware

### Automatically summarize the conversation when approaching the token limit

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain.agents import create_agent
from langchain.agent.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


## Agent with message Based summarization

agent=create_agent(

    model="gpt-5",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

agent

In [ ]:
# making thread id
config={"configurable":{"thread_id":"test1"}}

In [ ]:
# Alternate test data

questions=[
    "what is 2+2",
    "what is 16/2",
    "what is 5+2",
    "what is 8+2",
    "what is 2*4",
    "what is 2-5",

]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages":{response})

SyntaxError: unterminated string literal (detected at line 9) (1949325743.py, line 9)

In [ ]:
from langchain.agents import create_agent
from langchain.agent.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


@tool

def search_hostel(city:str)->str:
    """Search hotels return long responses"""
    return f"""hotels in {city}":
    1. Alharam
    2. Berrycrush
    3. Neelum View"""



agent=create_agent(

    model="gpt-5",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)


config={"configurable":{"thread_id":"test1"}}

# Human In Loop Middleware

- pause the  agent execution for human approval,editing or rejecting tool calls

In [ ]:
from langchain.agents import create_agent
from langchain.agent.middleware import HumanInLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """"Mock fuction to read the email by id"""
    return f"the content of email is{email_id}"



def send_email_tool(recepient:str,subject:str,body:str)->str:
    """mock function for sending email"""
    retun f"email send to{recepient}with subject{subject}"


In [ ]:
agent=create_agent(

    model="gpt-5",
    tools=[read_email_tool,send_email_tool]
    checkpointer=InMemorySaver(),
    middleware=[
     HumanInLoopMiddleware(
         interrupt_on={
            "send_email_tool":{
                 "allowed_decision":["approve","edit","reject"]
             }
            "read_email_tool":False
         }
     )
    ]
)

agent




# For Approving

In [ ]:
config={"configurable":{"thread_id":"test1"}}

result=agent.invoke({
    "messages":[HumanMessage(content="Send email to xyz with subject hi and body how are you")]
}, config=config)

In [ ]:
from langgraph.types import Command

# Step 1 approving

if "__interrupt__" in result:
    print("approve")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                            ]
            }
        )
        , config=config


    )
    print(f"Result{result["messages"][-1].content}")


# For Rejecting

In [ ]:
config={"configurable":{"thread_id":"test1"}}

result=agent.invoke({
    "messages":[HumanMessage(content="Send email to xyz with subject hi and body how are you")]
}, config=config)

In [ ]:
from langchain.agents import create_agent
from langchain.agent.middleware import HumanInLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """"Mock fuction to read the email by id"""
    return f"the content of email is{email_id}"



def send_email_tool(recepient:str,subject:str,body:str)->str:
    """mock function for sending email"""
    retun f"email send to{recepient}with subject{subject}"



agent=create_agent(

    model="gpt-5",
    tools=[read_email_tool,send_email_tool]
    checkpointer=InMemorySaver(),
    middleware=[
     HumanInLoopMiddleware(
         interrupt_on={
            "send_email_tool":{
                 "allowed_decision":["approve","edit","reject"]
             }
            "read_email_tool":False
         }
     ),
    ],
)

agent


In [ ]:
from langgraph.types import Command

# Step 1 approving

if "__interrupt__" in result:
    print("Paused_approving")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                            ]
            }
        )
        , config=config


    )
    print(f"Result{result["messages"][-1].content}")


# For Editing


In [ ]:
config={"configurable":{"thread_id":"test1"}}

result=agent.invoke({
    "messages":[HumanMessage(content="Send email to xyz with subject hi and body how are you")]
}, config=config)

In [ ]:
from langchain.agents import create_agent
from langchain.agent.middleware import HumanInLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """"Mock fuction to read the email by id"""
    return f"the content of email is{email_id}"



def send_email_tool(recepient:str,subject:str,body:str)->str:
    """mock function for sending email"""
    return f"email send to{recepient}with subject{subject}"



agent=create_agent(

    model="gpt-5",
    tools=[read_email_tool,send_email_tool]
    checkpointer=InMemorySaver(),
    middleware=[
     HumanInLoopMiddleware(
         interrupt_on={
            "send_email_tool":{
                 "allowed_decision":["approve","edit","reject"]
             }
            "read_email_tool":False
         }
     ),
    ],
)

agent